In [1]:
import glob
import pandas as pd
import numpy as np
import cv2
from PIL import Image
import sys
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import logging
from logging.handlers import RotatingFileHandler
sys.path.append("../../../../donut/src/test/")
from common.config import cfg

In [2]:
target_size=(cfg.image_width,cfg.image_height)
resized_image_save_path = Path("../../../data/image_resize/images")
scaling_annotation_save_path = Path("../../../data/image_resize/annotations")

target_size

(640, 640)

In [3]:
log_file = "./logs/ImageResize.log"
logging.basicConfig(
    handlers=[RotatingFileHandler(log_file, maxBytes=10 * 1024 * 1024, backupCount=3)],
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

In [4]:
# list all image paths
image_dir = "../../../data/raw/train/images/"
image_files = glob.glob(image_dir + "*")
print(len(image_files))
print(image_files[:5])

60578
['../../../data/raw/train/images/45df1fe3293b.jpg', '../../../data/raw/train/images/b2ab3b743d4e.jpg', '../../../data/raw/train/images/51d3b1a6baf3.jpg', '../../../data/raw/train/images/a9e9ce9277c1.jpg', '../../../data/raw/train/images/7f1f545fe081.jpg']


In [5]:
# list all annotation files paths

annotations_dir="../../../data/raw/train/annotations/"
annotations_files=glob.glob(annotations_dir+"*")
print(len(annotations_files))
print(annotations_files[:5])

60578
['../../../data/raw/train/annotations/e91e28111e86.json', '../../../data/raw/train/annotations/75c0449f6917.json', '../../../data/raw/train/annotations/66dd2a250237.json', '../../../data/raw/train/annotations/58595c30beab.json', '../../../data/raw/train/annotations/497a547454d7.json']


In [6]:
# drop unclean data from the list

# the bad list from ImageClean
bad_list = [
    "../../../data/raw/train/images/872d1be39bae.jpg",
    "../../../data/raw/train/images/3ef41bbc82c3.jpg",
    "../../../data/raw/train/images/73cfbba65962.jpg",
]

set_bad = set(bad_list)
image_files_filtered = [item for item in image_files if item not in set_bad]

print(len(image_files_filtered))

image_files=image_files_filtered

60575


In [7]:
# define the annotation scaling method


def annotation_resize(
    ori_image_path: str,
    origin_annotation_path: str,
    saved_path: str,
    resized_w_h: tuple = (512, 512),
):
    '''
    scaling the annotation json file values when the corresponding image size changes
    Params:
        ori_image_path: the path of original image
        origin_annotation_path: the path of the original annotation file
        saved path: the path where the updated annotation file should be saved
        resized_w_h: (width,height)
    '''

    origin_image = Image.open(fp=ori_image_path)
    width_ori, height_ori = origin_image.size

    width_new, height_new = resized_w_h

    scale_x = width_new / width_ori
    scale_y = height_new / height_ori

    with open(origin_annotation_path, "r") as f:
        annotation = json.load(f)

    # do not do modification on raw json file
    updated_annotation = annotation.copy()

    # update plot-bb
    updated_annotation["plot-bb"]["height"] = int(
        updated_annotation["plot-bb"]["height"] * scale_y
    )
    updated_annotation["plot-bb"]["width"] = int(
        updated_annotation["plot-bb"]["width"] * scale_x
    )
    updated_annotation["plot-bb"]["y0"] = int(
        updated_annotation["plot-bb"]["y0"] * scale_y
    )
    updated_annotation["plot-bb"]["x0"] = int(
        updated_annotation["plot-bb"]["x0"] * scale_x
    )
    # update text ticks
    for item in updated_annotation["text"]:
        polygon = item["polygon"]

        polygon["x0"] = int(polygon["x0"] * scale_x)
        polygon["x1"] = int(polygon["x1"] * scale_x)
        polygon["x2"] = int(polygon["x2"] * scale_x)
        polygon["x3"] = int(polygon["x3"] * scale_x)

        polygon["y0"] = int(polygon["y0"] * scale_y)
        polygon["y1"] = int(polygon["y1"] * scale_y)
        polygon["y2"] = int(polygon["y2"] * scale_y)
        polygon["y3"] = int(polygon["y3"] * scale_y)

    # axes
    for axis in ["x-axis", "y-axis"]:
        if (
            axis in updated_annotation["axes"]
            and "ticks" in updated_annotation["axes"][axis]
        ):
            for tick in updated_annotation["axes"][axis]["ticks"]:
                tick_pt = tick["tick_pt"]
                tick_pt["x"] = int(tick_pt["x"] * scale_x)
                tick_pt["y"] = int(tick_pt["y"] * scale_y)

    # elements
    for elements in updated_annotation["visual-elements"]:
        # print(elements)
        if len(updated_annotation["visual-elements"][elements]) != 0:
            if elements == "bars":
                for ticks in updated_annotation["visual-elements"][elements]:
                    # print(ticks)
                    ticks["width"] = ticks["width"] * scale_x
                    ticks["height"] = ticks["height"] * scale_y
                    ticks["x0"] = ticks["x0"] * scale_x
                    ticks["y0"] = ticks["y0"] * scale_y
            else:
                for ticks in updated_annotation["visual-elements"][elements]:
                    # print(ticks)
                    for tick in ticks:
                        # print(tick['x'])
                        tick["x"] = tick["x"] * scale_x
                        tick["y"] = tick["y"] * scale_y

    # data series
    for data in updated_annotation["data-series"]:
        if isinstance(data["x"], str):
            data["y"] = data["y"] * scale_y
        elif isinstance(data["y"], str):
            data["x"] = data["x"] * scale_x
        else:
            data["x"] = data["x"] * scale_x
            data["y"] = data["y"] * scale_y

    with open(saved_path, "w") as f:
        json.dump(updated_annotation, f, indent=4)

    return

In [8]:
# mark the image paths which finish the resizing successfully
resize_complete_pth=[]

for idx in range(len(image_files)):
    img_path=image_files[idx]
    image_id=img_path.split('/')[-1].split('.')[0]
    anno_path = [file for file in annotations_files if image_id in file]

    # reconstruct image path and annotation path
    image_saved_path=resized_image_save_path / f"{image_id}.jpg"
    annotation_saved_path=scaling_annotation_save_path / f"{image_id}.json"

    # resize image
    image = cv2.imread(str(img_path))
    resized_img = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)

    # save the file
    cv2.imwrite(str(image_saved_path), resized_img)

    # scaling annotation
    annotation_resize(
        ori_image_path=img_path,
        origin_annotation_path=anno_path[0],
        saved_path=annotation_saved_path,
        resized_w_h=target_size,
    )
    logging.info(f"Resizing complete: {image_id} -> {image_saved_path}")

    resize_complete_pth.append(img_path)



In [9]:
len(resize_complete_pth)

60575